In [1]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

C:\Users\Dell\anaconda3\envs\pytorch_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [3]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [4]:
train_data["dialogue"][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [5]:
train_data.sample(10)

,id,dialogue,summary
2959,13828814,Helen: He didn’t believe me\r\nKeesha: Whaaat ...,"Helen is pregnant. She will show him the test,..."
8398,13682211,Frank: Fancy going to see a film tonight?\r\nJ...,Jake and Frank will go and see the new Marvel ...
11766,13729221,Fiona: <file_photo>\r\nEliza: who's that hotti...,Fiona has changed her haircolor to blonde today.
3446,13728114,Jillian: I bought myself 3 dresses today!\r\nC...,Jillian bought three black dresses today.
5075,13717320,Brian: How's the renovation going?\r\nGillian:...,Renovation works at Gillian's and Brad's new a...
135,13681805,"Sean: Ruby, do you know where is the little, r...","Ruby tossed away the small, red pan, although ..."
2767,13829135,"Chandler: Okay, so I have a proposition to mak...",Chandler will try to write a one-shot only if ...
4350,13828036,Lindsay: did you see that movie i lent you?\r\...,Lindsay asked Raul to return her the movie she...
276,13681046,Maggie: What are you doing this weekend?\r\nLi...,Liz has to study this weekend but finds time t...
1510,13682054,John: Where are you going to go this summer?\r...,William and Ann are going on vacation to the A...


In [6]:
train_data.shape # take 4k samples

(14732, 3)

In [7]:
val_data.shape  # 500 samples

(818, 3)

In [8]:
#random sampling
#for saving our time further we can use entire train or val data

#train_data = train_data.sample(n = 14000, random_state = 42).reset_index(drop = True)
#val_data = val_data.sample(n = 450, random_state = 42).reset_index(drop = True)

In [9]:
train_data.shape

(14732, 3)

In [10]:
#Data preprocessing

import re
import pandas as pd

def clean_data(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = re.sub(r"\r\n", " ", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = text.strip()

    return text

In [11]:
print("Training missing values:")
print(train_data[["dialogue", "summary"]].isnull().sum())

print("\nValidation missing values:")
print(val_data[["dialogue", "summary"]].isnull().sum())

Training missing values:
dialogue    1
summary     0
dtype: int64

Validation missing values:
dialogue    0
summary     0
dtype: int64


In [12]:
train_data = train_data.dropna(subset=["dialogue"]).copy()

In [13]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data) 
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data) 
val_data["summary"] = val_data["summary"].apply(clean_data)


In [14]:
train_data["dialogue"][0]

"Amanda: I baked cookies. Do you want some? Jerry: Sure! Amanda: I'll bring you tomorrow :-)"

In [15]:
#Tokenization

tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [16]:
# row dta => tokenized inputs for fine-tuning  i.e text to token_ids

def tokenize(row):
    dialogue = "summarize: " + row["dialogue"]

    # Tokenize the input dialogue
    inputs = tokenizer(
        dialogue,
        max_length=512,
        truncation=True,
        padding=False
    )

    # Tokenize the reference summary
    targets = tokenizer(
        text_target=row["summary"],
        max_length=150,
        truncation=True,
        padding=False
    )

    # Target token IDs become training labels
    inputs["labels"] = targets["input_ids"]

    return inputs

In [17]:
#train_dataset= train_data.apply(tokenize, axis = 1).tolist()
#val_dataset = val_data.apply(tokenize, axis = 1).tolist()

train_dataset = [
    tokenize(row)
    for _, row in train_data.iterrows()
]

val_dataset = [
    tokenize(row)
    for _, row in val_data.iterrows()
]

print("Training examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))

Training examples: 14731
Validation examples: 818


In [18]:
train_dataset[0]

{'input_ids': [21603, 10, 21542, 10, 27, 13635, 5081, 5, 531, 25, 241, 128, 58, 16637, 10, 10625, 55, 21542, 10, 27, 31, 195, 830, 25, 5721, 3, 10, 18, 61, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [21542, 13635, 5081, 11, 56, 830, 16637, 128, 5721, 5, 1]}

In [19]:
#input ids = dialogue => token ids

# 1 => end of sequence, 0=> padding

#attention mask  #checks the validy for ids in ip ids and return 1 for valid ids else 0
#labels = tokens => summary token

In [20]:
#working with our model
 #nlp => generation task

import torch

from transformers import (
    T5ForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments
)

model = T5ForConditionalGeneration.from_pretrained("t5-small")

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 330.19it/s]


In [21]:
#fine-tune

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device: ", device)
model.to(device)

device:  cpu


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [22]:
#training arguments

training_args = TrainingArguments(
    output_dir="./results_full_data",

    num_train_epochs=3,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    learning_rate=5e-5,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    save_total_limit=2,
    logging_steps=100,

    report_to="none"
)

In [23]:
trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    data_collator=data_collator
)

In [24]:
#train the model
trainer.train()

C:\Users\Dell\anaconda3\envs\pytorch_env\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,1.979024,1.788965
2,1.884828,1.745137
3,1.855476,1.735078


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.29it/s]
C:\Users\Dell\anaconda3\envs\pytorch_env\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.78it/s]
C:\Users\Dell\anaconda3\envs\pytorch_env\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.15it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=5526, training_loss=1.9514700012538384, metrics={'train_runtime': 58448.3864, 'train_samples_per_second': 0.756, 'train_steps_per_second': 0.095, 'total_flos': 3812996996825088.0, 'train_loss': 1.9514700012538384, 'epoch': 3.0})

In [25]:
#save trained model
model.save_pretrained("./saved_summary_model_v2")
#save tokenizer 
tokenizer.save_pretrained("./saved_summary_model_v2")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.54it/s]


('./saved_summary_model_v2\\tokenizer_config.json',
 './saved_summary_model_v2\\tokenizer.json')

In [26]:
eval_results = trainer.evaluate()

print("Evaluation Results:")
print(eval_results)

C:\Users\Dell\anaconda3\envs\pytorch_env\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch
1.855476,1.735078,3


Evaluation Results:
{'eval_loss': 1.7350776195526123}


In [27]:
#suppose we want to use this save model in future then simply use
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model_v2")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model_v2")

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 386.04it/s]


test the core logic for summarization

In [28]:
def summarize_dialogue(dialogue):

    dialogue = clean_data(dialogue)

    # Add task prefix
    dialogue = "summarize: " + dialogue

    # Tokenize
    inputs = tokenizer(
        dialogue,
        padding="max_length",
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    # Generate summary
    model.to(device)
    model.eval()

    with torch.no_grad():
        targets = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=150,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3
        )

    # Decode output
    summary = tokenizer.decode(
        targets[0],
        skip_special_tokens=True
    )

    return summary

In [29]:
test_dialogue = """ 
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""

summary = summarize_dialogue(test_dialogue)
print("summary: ", summary)

summary:  Artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. Expert concerns about bias in AI models, as they often reflect the data they are trained on.


In [31]:
idx = 0

test_dialogue = val_data["dialogue"].iloc[idx]
actual_summary = val_data["summary"].iloc[idx]

generated_summary = summarize_dialogue(test_dialogue)

print("Original Dialogue:")
print(test_dialogue)

print("\nActual Summary:")
print(actual_summary)

print("\nGenerated Summary:")
print(generated_summary)

Original Dialogue:
A: Hi Tom, are you busy tomorrow’s afternoon? B: I’m pretty sure I am. What’s up? A: Can you go with me to the animal shelter?. B: What do you want to do? A: I want to get a puppy for my son. B: That will make him so happy. A: Yeah, we’ve discussed it many times. I think he’s ready now. B: That’s good. Raising a dog is a tough issue. Like having a baby ;-) A: I'll get him one of those little dogs. B: One that won't grow up too big;-) A: And eat too much;-)) B: Do you know which one he would like? A: Oh, yes, I took him there last Monday. He showed me one that he really liked. B: I bet you had to drag him away. A: He wanted to take it home right away ;-). B: I wonder what he'll name it. A: He said he’d name it after his dead hamster – Lemmy - he's a great Motorhead fan :-)))

Actual Summary:
A will go to the animal shelter tomorrow to get a puppy for her son. They already visited the shelter last Monday and the son chose the puppy.

Generated Summary:
A wants to get a